<a href="https://colab.research.google.com/github/raffmca/AlgoTradeSample/blob/master/stocks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install yfinance scipy pandas numpy

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import norm
import matplotlib.pyplot as plt

# 1. CORE TRACKING LIST + DYNAMIC VOL SCANNER SETUP
# Base index-tracking tickers
tickers = ["SPY", "QQQ", "^SPX"]  # Note: ^SPX is used for Cboe S&P 500 Index options

# Dynamic Mega-Cap Unusual Volume Pull:
# We check a pool of standard high-gamma retail mega-cap drivers.
# The loop below compares their current session volume against their 3-month daily average.
mega_cap_pool = ["NVDA", "TSLA", "AAPL", "MSFT", "AMZN", "META", "AMD", "MU", "GOOG","AVGO"]

print("Scanning for Mega-Caps with Unusual Volume Catalyst...")
for ticker in mega_cap_pool:
    try:
        stock = yf.Ticker(ticker)
        # Pull today's data vs average daily volume
        history = stock.history(period="5d")
        if history.empty:
            continue

        current_volume = history['Volume'].iloc[-1]
        info = stock.info
        avg_volume = info.get('averageVolume', 1)

        # Calculate Relative Volume ratio (similar to TOS unusual volume logic)
        # Filters for tokens pacing significantly above baseline context
        rel_vol = current_volume / (avg_volume / 5) # Est. intraday pacing

        if rel_vol >= 1.15 or current_volume > (avg_volume * 0.5):
            print(f"🔥 Added {ticker} to Watchlist (Unusual Volume Pacing: {rel_vol:.2f}x)")
            if ticker not in tickers:
                tickers.append(ticker)
    except Exception as e:
        # Fallback to ensure common hyper-liquid retail names populate if API constraints hit
        if ticker in ["NVDA", "TSLA"] and ticker not in tickers:
            tickers.append(ticker)

# 2. ITERATIVE EXPOSURE ENGINE
r = 0.05      # 5% Risk-free rate
t = 1 / 365   # ~1 Day DTE structure

def calculate_greeks_and_exposure(df, spot, is_call=True):
    results = []
    for _, row in df.iterrows():
        K = row['strike']
        oi = row['openInterest']
        vol = row['impliedVolatility']
        if oi <= 0 or vol <= 0: continue

        d1 = (np.log(spot / K) + (r + 0.5 * vol ** 2) * t) / (vol * np.sqrt(t))
        d2 = d1 - vol * np.sqrt(t)

        gamma = norm.pdf(d1) / (spot * vol * np.sqrt(t))
        vanna = -norm.pdf(d1) * d2 / vol

        # Scaling framework relative to contract sizes and basic price profiles
        multiplier = 100 if spot < 1000 else 10 # Adjust for SPX cash indexing sizes
        gex = gamma * oi * multiplier * (1 if is_call else -1) * spot * 0.01
        vex = vanna * oi * multiplier * (1 if is_call else -1)

        results.append({'Strike': K, 'GEX': gex, 'VEX': vex})
    return pd.DataFrame(results)

# Loop and compile every asset down into individual structural maps
for ticker in tickers:
    try:
        print(f"\nProcessing Options Market Profile for: {ticker}...")
        inst = yf.Ticker(ticker)

        # Resolve Index cash close vs underlying asset close
        hist_data = inst.history(period="1d")
        if hist_data.empty:
            print(f"⚠️ Could not pull spot price for {ticker}. Skipping.")
            continue
        spot_price = hist_data["Close"].iloc[-1]

        expirations = inst.options
        if not expirations: continue
        target_exp = expirations[0] # Isolate the front-month 0DTE/1DTE slice

        opt_chain = inst.option_chain(target_exp)

        calls_df = calculate_greeks_and_exposure(opt_chain.calls, spot_price, is_call=True)
        puts_df = calculate_greeks_and_exposure(opt_chain.puts, spot_price, is_call=False)

        if calls_df.empty or puts_df.empty: continue

        full_chain = pd.concat([calls_df, puts_df]).groupby('Strike').sum().reset_index()

        # Isolate exactly inside your 5% Strike range band around the asset spot
        filtered_chain = full_chain[(full_chain['Strike'] >= spot_price * 0.975) & (full_chain['Strike'] <= spot_price * 1.025)]

        if filtered_chain.empty: continue

        call_wall = filtered_chain.loc[filtered_chain['GEX'].idxmax()]
        put_wall = filtered_chain.loc[filtered_chain['GEX'].idxmin()]
        gamma_flip = filtered_chain.loc[filtered_chain['GEX'].abs().idxmin()]

        # Print Structural Breakdowns
        print("-" * 60)
        print(f"📊 {ticker} STRUCTURE | Spot: ${spot_price:.2f} | Exp: {target_exp}")
        print(f"🟢 CALL WALL: ${call_wall['Strike']:.2f}  | 🔴 PUT WALL: ${put_wall['Strike']:.2f}  | ⚡ FLIP: ${gamma_flip['Strike']:.2f}")
        print("-" * 60)

        # Plot structural layouts
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
        colors_gex = ['green' if x > 0 else 'red' for x in filtered_chain['GEX']]
        ax1.barh(filtered_chain['Strike'], filtered_chain['GEX'], color=colors_gex, alpha=0.5, edgecolor='black')
        ax1.axhline(spot_price, color='blue', linestyle='--', linewidth=1.5, label='Spot')
        ax1.axhline(call_wall['Strike'], color='green', linestyle=':', linewidth=1.5)
        ax1.axhline(put_wall['Strike'], color='red', linestyle=':', linewidth=1.5)
        ax1.set_title(f"{ticker} GEX Profiles")
        ax1.grid(True, alpha=0.2)

        colors_vex = ['blue' if x > 0 else 'purple' for x in filtered_chain['VEX']]
        ax2.barh(filtered_chain['Strike'], filtered_chain['VEX'], color=colors_vex, alpha=0.5, edgecolor='black')
        ax2.axhline(spot_price, color='blue', linestyle='--')
        ax2.set_title(f"{ticker} VEX Profiles")
        ax2.grid(True, alpha=0.2)

        plt.tight_layout()
        plt.show()

    except Exception as error:
        print(f"❌ Error compiling mapping for {ticker}: {error}")

Scanning for Mega-Caps with Unusual Volume Catalyst...
🔥 Added NVDA to Watchlist (Unusual Volume Pacing: 3.37x)
🔥 Added TSLA to Watchlist (Unusual Volume Pacing: 4.35x)
🔥 Added AAPL to Watchlist (Unusual Volume Pacing: 4.45x)
🔥 Added MSFT to Watchlist (Unusual Volume Pacing: 2.48x)
🔥 Added AMZN to Watchlist (Unusual Volume Pacing: 3.56x)
🔥 Added META to Watchlist (Unusual Volume Pacing: 4.56x)
🔥 Added AMD to Watchlist (Unusual Volume Pacing: 3.26x)
🔥 Added MU to Watchlist (Unusual Volume Pacing: 2.73x)
🔥 Added GOOG to Watchlist (Unusual Volume Pacing: 3.02x)
🔥 Added AVGO to Watchlist (Unusual Volume Pacing: 6.68x)

Processing Options Market Profile for: SPY...

Processing Options Market Profile for: QQQ...

Processing Options Market Profile for: ^SPX...

Processing Options Market Profile for: NVDA...

Processing Options Market Profile for: TSLA...

Processing Options Market Profile for: AAPL...

Processing Options Market Profile for: MSFT...

Processing Options Market Profile for: AMZN

In [ ]:
pip install tabulate

In [ ]:
import pandas as pd
import yfinance as yf
from tabulate import tabulate


def fetch_unusual_options(
    tickers, min_volume=1000, min_vol_oi_ratio=2.0, min_dollar_val=100000
):
    """Scans tickers for options with unusual volume relative to open interest.

    Calculates average contract price and total dollar value.
    """
    records = []

    for symbol in tickers:
        try:
            ticker = yf.Ticker(symbol)
            expirations = ticker.options

            # Scan up to the first 4 nearest expiration dates for speed
            for expiry in expirations[:4]:
                chain = ticker.option_chain(expiry)

                # Process both Calls and Puts
                for option_type, df in [
                    ("CALL", chain.calls),
                    ("PUT", chain.puts),
                ]:
                    if df.empty:
                        continue

                    for _, row in df.iterrows():
                        vol = row.get("volume", 0)
                        oi = row.get("openInterest", 0)

                        # Skip low volume or invalid data
                        if pd.isna(vol) or vol < min_volume:
                            continue

                        # Treat OI of 0 as 1 to avoid division by zero
                        adj_oi = oi if (not pd.isna(oi) and oi > 0) else 1
                        vol_oi_ratio = vol / adj_oi

                        # Filter for high Vol / OI ratio
                        if vol_oi_ratio >= min_vol_oi_ratio:
                            bid = row.get("bid", 0)
                            ask = row.get("ask", 0)
                            last_price = row.get("lastPrice", 0)

                            # Estimate Average Price (Midpoint of Bid/Ask, fallback to Last Price)
                            if bid > 0 and ask > 0:
                                avg_price = (bid + ask) / 2.0
                            else:
                                avg_price = last_price

                            # Total Dollar Value = Volume * Average Price * 100 multiplier
                            total_dollar_val = vol * avg_price * 100

                            # Filter out small dollar value noise
                            if total_dollar_val < min_dollar_val:
                                continue

                            # Infer trade context based on execution relative to mid-price
                            if last_price >= ask and ask > 0:
                                context = "Ask Side (Bullish Buy Sweep)"
                            elif last_price <= bid and bid > 0:
                                context = "Bid Side (Bearish Sell Sweep)"
                            else:
                                context = "Mid / Block Trade"

                            records.append(
                                {
                                    "TickerType": f"{symbol} {option_type}",
                                    "Strike Price": f"${row['strike']:.2f}",
                                    "Expiration Date": expiry,
                                    "Volume": int(vol),
                                    "Prior OI": int(oi) if pd.notna(oi) else 0,
                                    "Volume / OI": round(vol_oi_ratio, 2),
                                    "Average Price": f"${avg_price:.2f}",
                                    "Total Dollar Value": f"${total_dollar_val:,.2f}",
                                    "Execution Side / Context": context,
                                }
                            )
        except Exception as e:
            print(f"Could not fetch data for {symbol}: {e}")

    # Build DataFrame
    df_results = pd.DataFrame(records)

    # Sort results by highest Volume / OI ratio
    if not df_results.empty:
        df_results = df_results.sort_values(
            by="Volume / OI", ascending=False
        ).reset_index(drop=True)

    return df_results


# --- EXECUTION ---
if __name__ == "__main__":
    # Watchlist of target stocks
    watchlist = ["MRNA", "RIOT", "AVGO", "WMT", "SEDG", "CRML", "LIN", "CTVA"]

    print("Scanning for unusual options activity...\n")
    results = fetch_unusual_options(
        tickers=watchlist,
        min_volume=500,  # Min contracts traded
        min_vol_oi_ratio=1.5,  # Volume must be at least 1.5x Open Interest
        min_dollar_val=50000,  # Min total position premium ($50k)
    )

    if not results.empty:
        print(tabulate(results, headers="keys", tablefmt="github", showindex=False))
    else:
        print("No unusual options activity matching the criteria was found.")

Scanning for unusual options activity...

| TickerType   | Strike Price   | Expiration Date   |   Volume |   Prior OI |   Volume / OI | Average Price   | Total Dollar Value   | Execution Side / Context      |
|--------------|----------------|-------------------|----------|------------|---------------|-----------------|----------------------|-------------------------------|
| WMT CALL     | $104.00        | 2026-09-04        |     1043 |          0 |       1043    | $1.97           | $205,471.00          | Bid Side (Bearish Sell Sweep) |
| WMT CALL     | $106.00        | 2026-08-28        |     2624 |          3 |        874.67 | $0.80           | $209,920.00          | Mid / Block Trade             |
| WMT CALL     | $103.00        | 2026-08-28        |     1333 |          2 |        666.5  | $1.91           | $253,936.50          | Bid Side (Bearish Sell Sweep) |
| WMT CALL     | $104.00        | 2026-08-21        |     4698 |          8 |        587.25 | $0.79           | $368,793.00

In [ ]:
import datetime
import pandas as pd
import yfinance as yf
from tabulate import tabulate


def generate_thesis(
    symbol, option_type, strike, underlying_price, expiry_str, side, vol_oi_ratio, total_val
):
    """Generates a qualitative thesis/context narrative based on contract mechanics."""
    today = datetime.date.today()
    expiry_date = datetime.datetime.strptime(expiry_str, "%Y-%m-%d").date()
    days_to_expiry = (expiry_date - today).days

    # 1. Determine Expiry Horizon
    if days_to_expiry <= 14:
        horizon = "ultra short-dated"
    elif days_to_expiry <= 60:
        horizon = "medium-term"
    elif days_to_expiry <= 180:
        horizon = "intermediate"
    else:
        horizon = "long-dated LEAP"

    # 2. Determine Moneyness
    pct_diff = (strike - underlying_price) / underlying_price
    if abs(pct_diff) < 0.02:
        moneyness = "at-the-money"
    elif option_type == "CALL":
        moneyness = "out-of-the-money" if pct_diff > 0 else "in-the-money"
    else:
        moneyness = "out-of-the-money" if pct_diff < 0 else "in-the-money"

    # 3. Synthesize Strategic Thesis
    if option_type == "CALL":
        if "Ask" in side:
            if days_to_expiry <= 14:
                thesis = f"Aggressive {horizon} momentum sweep targeting immediate upside momentum."
            elif moneyness == "out-of-the-money":
                thesis = f"High-conviction {horizon} bullish call accumulation targeting breakout past ${strike:.2f}."
            else:
                thesis = f"Directional {horizon} call buying driving high-delta leveraged leverage."
        elif "Bid" in side:
            thesis = f"Institutional covered call writing or profit-taking against existing long position."
        else:
            thesis = f"Large institutional block execution establishing structured upside exposure."
    else:  # PUT
        if "Ask" in side:
            if days_to_expiry <= 14:
                thesis = f"Tactical {horizon} protective hedge or downside speculation ahead of short-term catalysts."
            else:
                thesis = f"Institutional downside hedge or macro protective positioning against broader pullback risk."
        elif "Bid" in side:
            if days_to_expiry > 180:
                thesis = f"Ultra long-dated institutional LEAP put selling / cash-secured yield generation."
            else:
                thesis = f"Strategic put selling / spread financing to acquire shares at a lower cost basis."
        else:
            thesis = f"Multi-leg institutional protective collar or portfolio risk-reversal execution."

    return thesis


def fetch_unusual_options(
    tickers, min_volume=1000, min_vol_oi_ratio=2.0, min_dollar_val=100000
):
    records = []

    for symbol in tickers:
        try:
            ticker = yf.Ticker(symbol)
            expirations = ticker.options

            # Get current stock price for moneyness calculation
            fast_info = ticker.fast_info
            underlying_price = fast_info.get('lastPrice', 0) or fast_info.get('previousClose', 0)

            # Scan up to the first 5 expiration dates
            for expiry in expirations[:5]:
                chain = ticker.option_chain(expiry)

                for option_type, df in [("CALL", chain.calls), ("PUT", chain.puts)]:
                    if df.empty:
                        continue

                    for _, row in df.iterrows():
                        vol = row.get("volume", 0)
                        oi = row.get("openInterest", 0)

                        if pd.isna(vol) or vol < min_volume:
                            continue

                        adj_oi = oi if (not pd.isna(oi) and oi > 0) else 1
                        vol_oi_ratio = vol / adj_oi

                        if vol_oi_ratio >= min_vol_oi_ratio:
                            bid = row.get("bid", 0)
                            ask = row.get("ask", 0)
                            last_price = row.get("lastPrice", 0)

                            if bid > 0 and ask > 0:
                                avg_price = (bid + ask) / 2.0
                            else:
                                avg_price = last_price

                            total_dollar_val = vol * avg_price * 100

                            if total_dollar_val < min_dollar_val:
                                continue

                            # Classify execution side
                            if last_price >= ask and ask > 0:
                                side = "Ask Side"
                            elif last_price <= bid and bid > 0:
                                side = "Bid Side"
                            else:
                                side = "Mid / Block"

                            # Generate qualitative thesis/explanatory reason
                            thesis = generate_thesis(
                                symbol=symbol,
                                option_type=option_type,
                                strike=row['strike'],
                                underlying_price=underlying_price,
                                expiry_str=expiry,
                                side=side,
                                vol_oi_ratio=vol_oi_ratio,
                                total_val=total_dollar_val
                            )

                            records.append(
                                {
                                    "TickerType": f"{symbol} {option_type}",
                                    "Strike Price": f"${row['strike']:.2f}",
                                    "Expiration Date": expiry,
                                    "Volume": int(vol),
                                    "Prior OI": int(oi) if pd.notna(oi) else 0,
                                    "Volume / OI": round(vol_oi_ratio, 2),
                                    "Average Price": f"${avg_price:.2f}",
                                    "Total Dollar Value": f"${total_dollar_val:,.2f}",
                                    "Execution Side": side,
                                    "Explanatory Thesis / Context": thesis,
                                }
                            )
        except Exception as e:
            print(f"Could not fetch data for {symbol}: {e}")

    df_results = pd.DataFrame(records)

    if not df_results.empty:
        df_results = df_results.sort_values(by="Volume / OI", ascending=False).reset_index(drop=True)

    return df_results


# --- EXECUTION ---
if __name__ == "__main__":
    watchlist = ["MRNA", "RIOT", "AVGO", "WMT", "SEDG", "LIN", "CTVA", "ICE", "IT"]

    print("Scanning market for unusual options flow and generating thesis context...\n")
    results = fetch_unusual_options(
        tickers=watchlist,
        min_volume=500,
        min_vol_oi_ratio=1.5,
        min_dollar_val=750000,
    )

    if not results.empty:
        print(tabulate(results, headers="keys", tablefmt="github", showindex=False))
    else:
        print("No unusual options activity matching the criteria was found.")

Scanning market for unusual options flow and generating thesis context...

| TickerType   | Strike Price   | Expiration Date   |   Volume |   Prior OI |   Volume / OI | Average Price   | Total Dollar Value   | Execution Side   | Explanatory Thesis / Context                                                                       |
|--------------|----------------|-------------------|----------|------------|---------------|-----------------|----------------------|------------------|----------------------------------------------------------------------------------------------------|
| MRNA CALL    | $157.50        | 2026-09-18        |     4661 |        237 |         19.67 | $2.23           | $1,041,733.50        | Mid / Block      | Large institutional block execution establishing structured upside exposure.                       |
| AVGO PUT     | $510.00        | 2026-09-18        |     1245 |        159 |          7.83 | $144.45         | $17,984,025.00       | Ask Side         | Tactic

In [ ]:
!pip install streamlit plotly pandas numpy -q
!npm install -g localtunnel -q



⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏
added 22 packages in 3s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏npm notice
npm notice New major version of npm available! 10.8.2 -> 12.0.2
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.0.2
npm notice To update run: npm install -g npm@12.0.2
npm notice
⠏

In [ ]:
%%writefile app.py
import math
import random
import time
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import streamlit as st

st.set_page_config(
    page_title="QUANT_TERMINAL // OFI Engine",
    page_icon="⚡",
    layout="wide",
)

st.title("⚡ MICROSTRUCTURE & ORDER FLOW TERMINAL")
st.success("App running directly in Google Colab ephemeral environment (`/content/app.py`).")

# Simple live metric demonstration
st.metric("Engine Status", "ACTIVE", "0DTE Stream Ready")

Overwriting app.py


In [ ]:
!curl https://ipv4.icanhazip.com
!streamlit run app.py & npx localtunnel --port 8501

34.81.94.8
⠙⠹

⠸⠼⠴⠦2026-09-11 16:19:50.600 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.81.94.8:8501

your url is: https://late-suits-admire.loca.lt
  Stopping...
^C


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb



Selecting previously unselected package cloudflared.
(Reading database ... 122797 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.9.1) ...
Setting up cloudflared (2026.9.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [ ]:
!pkill -f streamlit
!pkill -f cloudflared

In [ ]:
!streamlit run app.py --server.port 8501 > /dev/null 2>&1 &

In [ ]:
!cloudflared tunnel --url http://localhost:8501

2026-09-11T16:25:46Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-11T16:25:46Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-11T16:25:50Z INF +--------------------------------------------------------------------------------------------+
2026-09-11T16:25:50Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-11T16:25:50Z INF |  https://nine-possession-phoenix-enclosed.trycloudflar

In [ ]:
!python /content/drive/MyDrive/VSA_Project/vsa_runner.py

🚀 INITIALIZING SYNCHRONIZED PARALLEL VSA SCANNER: 503 TICKERS
[DIRECTORY LOG] Persistent Google Drive Location: '/content/drive/MyDrive/VSA_Project/09122026'
[CHART CONFIG] Option to switch off PNG generated: False

Launching synchronized concurrent thread extraction worker queues...
    ✨ TRIGGER FOUND: AON marked for Daily Distribution
    ✨ TRIGGER FOUND: BALL marked for Daily Distribution
    ✨ TRIGGER FOUND: CPRT marked for Daily Distribution
    ✨ TRIGGER FOUND: ETR marked for Daily Distribution
    ✨ TRIGGER FOUND: FICO marked for Weekly Accumulation
    ✨ TRIGGER FOUND: HPE marked for Weekly Accumulation
    ✨ TRIGGER FOUND: INVH marked for Daily Distribution
    ✨ TRIGGER FOUND: IR marked for Daily Distribution
    ✨ TRIGGER FOUND: KIM marked for Daily Distribution
    ✨ TRIGGER FOUND: ORCL marked for Daily Distribution
    ✨ TRIGGER FOUND: SWKS marked for Weekly Accumulation
    ✨ TRIGGER FOUND: UNH marked for Daily Distribution

Scan operations completed in 0.5 minutes.

🔥 V

In [ ]:
!pip install -q streamlit
!npm install -q -g localtunnel

⠙⠹⠸⠼⠴
changed 22 packages in 622ms
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴

In [ ]:
import os
os.makedirs('.streamlit', exist_ok=True)

with open('.streamlit/config.toml', 'w') as f:
    f.write("""
[theme]
primaryColor = "#00FF00"         # Neon Green (Bullish / Selected)
backgroundColor = "#000000"      # Pure Black Ground
secondaryBackgroundColor = "#1A1A1A" # Dark Charcoal Sidebar
textColor = "#FFFFFF"            # White text
font = "monospace"               # Monospace font fits data tables perfectly
""")

print("✓ Bloomberg Terminal theme written. Rerun your app link or refresh to see it!")


✓ Bloomberg Terminal theme written. Rerun your app link or refresh to see it!


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import plotly.graph_objects as go

st.set_page_config(page_title="Quant Analytics prof", layout="wide")

# Left Pane Controls
with st.sidebar:
    st.header("🎛️ Control Pane")
    strategy = st.selectbox("Trading Strategy", ["Mean Reversion (RSI)", "Trend  (MACD)"])
    ticker = st.text_input("Ticker Symbol", value="AAPL")
    run_btn = st.button("🚀 Run Backtest", use_container_width=True, type="primary")

# Main Content Pane
st.title(f"📊 Quant Analytics: {strategy}")
st.caption(f"Analyzing {ticker.upper()}")

# Row 1: KPI Metrics
col1, col2, col3 = st.columns(3)
with col1: st.metric(label="Total Return", value="+24.8%", delta="vs SPY +12.3%")
with col2: st.metric(label="Sharpe Ratio", value="1.84")
with col3: st.metric(label="Max Drawdown", value="-4.2%")

# Row 2: Simulated Chart (Plotly matches dark theme natively)
dates = pd.date_range(start="2026-01-01", periods=40, freq="B")
close_prices = 150 + np.cumsum(np.random.normal(0.5, 3, 40))
fig = go.Figure(data=[go.Candlestick(x=dates, open=close_prices, high=close_prices+2, low=close_prices-2, close=close_prices)])
fig.update_layout(template="plotly_dark", height=350, margin=dict(l=20, r=20, t=10, b=10), paper_bgcolor="rgba(0,0,0,0)")
st.plotly_chart(fig, use_container_width=True)


Overwriting app.py


In [ ]:
# 1. Install Ngrok python wrapper
!pip install -q pyngrok

# 2. Authenticate Ngrok (PASTE YOUR COPIED AUTHTOKEN BETWEEN THE QUOTES BELOW)
!ngrok config add-authtoken 3JBni8VJ7ABqqtt2tAV2mKqugTW_2ryWHugrYFXWbPVytEa1n

# 3. Securely start the web tunnel
from pyngrok import ngrok
import os

# Clear any old lingering tunnels
ngrok.kill()

# Open a secure public link pointing to Streamlit's port
public_url = ngrok.connect(8501)
print(f"\n🚀 CLICK THIS SECURE NGROK LINK TO OPEN YOUR DASHBOARD:\n{public_url.public_url}\n")

# 4. Start the Streamlit application server in the background
!streamlit run app.py --server.port 8501 > /dev/null 2>&1 &


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml

🚀 CLICK THIS SECURE NGROK LINK TO OPEN YOUR DASHBOARD:
https://ripening-tannery-unreached.ngrok-free.dev



In [ ]:
!python /content/drive/MyDrive/VSA_Project/vsa_runner.py

🚀 INITIALIZING SYNCHRONIZED PARALLEL VSA SCANNER: 503 TICKERS
[DIRECTORY LOG] Persistent Google Drive Location: '/content/drive/MyDrive/VSA_Project/09122026'
[CHART CONFIG] Option to switch off PNG generated: False

Launching synchronized concurrent thread extraction worker queues...
    ✨ TRIGGER FOUND: AON marked for Daily Distribution
    ✨ TRIGGER FOUND: BALL marked for Daily Distribution
    ✨ TRIGGER FOUND: CPRT marked for Daily Distribution
    ✨ TRIGGER FOUND: ETR marked for Daily Distribution
    ✨ TRIGGER FOUND: FICO marked for Weekly Accumulation
    ✨ TRIGGER FOUND: HPE marked for Weekly Accumulation
    ✨ TRIGGER FOUND: INVH marked for Daily Distribution
    ✨ TRIGGER FOUND: IR marked for Daily Distribution
    ✨ TRIGGER FOUND: KIM marked for Daily Distribution
    ✨ TRIGGER FOUND: ORCL marked for Daily Distribution
    ✨ TRIGGER FOUND: SWKS marked for Weekly Accumulation
    ✨ TRIGGER FOUND: UNH marked for Daily Distribution

Scan operations completed in 0.5 minutes.

📈 S

In [ ]:
import math
import numpy as np
import pandas as pd
import yfinance as yf
import requests
from scipy.stats import norm

# ==========================================
# 🔐 TRADIER LIVE CONFIGURATION
# ==========================================
# Paste your real production token here
TRADIER_ACCESS_TOKEN = "iQIDbxph7jNACgHbsMlXm00A1o2c"
# Enforcing the official production domain from your curl example
TRADIER_BASE_URL = "https://api.tradier.com/v1"

def get_tradier_overnight_spot(symbol="SPY"):
    """
    Fetches real-time extended-hours/overnight spot price from Tradier Live.
    """
    headers = {
        'Authorization': f'Bearer {TRADIER_ACCESS_TOKEN}',
        'Accept': 'application/json'
    }

    # Builds the exact path matching the working curl documentation
    url = f"{TRADIER_BASE_URL}/markets/quotes"
    params = {
        'symbols': symbol,
        'greeks': 'false'  # Kept false to accelerate server response times
    }

    try:
        response = requests.get(url, headers=headers, params=params)
        if response.status_code == 200:
            data = response.json()
            if 'quotes' in data and 'quote' in data['quotes']:
                quote = data['quotes']['quote']

                # Tradier returns a list if multiple symbols are passed,
                # or a dict for a single asset query.
                if isinstance(quote, list):
                    quote = quote[0]

                return float(quote['last'])
            else:
                print("⚠️ Tradier returned an empty quote payload.")
        elif response.status_code == 401:
            print("⚠️ Tradier Authorization Error (401). Verify your token string is correct.")
        else:
            print(f"⚠️ Tradier API failed with status {response.status_code}. URL: {response.url}")

    except Exception as e:
        print(f"⚠️ Error reaching Tradier: {e}")

    print("🔄 Defaulting to yfinance.")
    return None

def black_scholes_greeks(S, K, T, r, sigma):
    """Calculates the exact Gamma and Vanna for a specific strike."""
    if T <= 0 or sigma <= 0 or S <= 0:
        return 0.0, 0.0

    d1 = (math.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)

    gamma = norm.pdf(d1) / (S * sigma * math.sqrt(T))
    vanna = -norm.pdf(d1) * (d2 / sigma)

    return gamma, vanna

def fetch_and_calculate_gex(ticker_symbol="SPY"):
    print(f"⚡ Initializing Hybrid Data Stream for {ticker_symbol}...")

    # ─── STEP 1: RESOLVE OVERNIGHT SPOT PRICE VIA TRADIER LIVE ──────────
    spot_price = get_tradier_overnight_spot(ticker_symbol)

    # ─── STEP 2: FALLBACK TO YFINANCE IF TRADIER LOGINS FAIL ─────────────
    ticker = yf.Ticker(ticker_symbol)
    if spot_price is None:
        try:
            spot_price = ticker.history(period="1d", interval="1m")['Close'].iloc[-1]
            print(f"🔄 Using yfinance Fallback Spot Price.")
        except Exception:
            spot_price = ticker.info.get('regularMarketPrice', 760.58)
            print(f"🔄 Using yfinance Asset Info Fallback Spot Price.")

    print(f"🎯 Active Spot Price Locked: ${spot_price:.2f}")

    # ─── STEP 3: OPTIONS DATA FROM YFINANCE ─────────────────────────────
    print(f"📦 Pulling massive Options Open Interest profile via yfinance...")
    risk_free_rate = 0.045
    expirations = ticker.options[:5]

    all_strikes_data = []

    for exp_date in expirations:
        try:
            opt_chain = ticker.option_chain(exp_date)
        except Exception:
            continue

        time_to_exp = (pd.Timestamp(exp_date) - pd.Timestamp.now()).days / 365.25
        if time_to_exp <= 0:
            time_to_exp = 0.001

        # Process Calls
        for _, row in opt_chain.calls.iterrows():
            if pd.isna(row['impliedVolatility']) or not row['openInterest']: continue
            gamma, vanna = black_scholes_greeks(spot_price, row['strike'], time_to_exp, risk_free_rate, row['impliedVolatility'])
            all_strikes_data.append({
                'Strike': row['strike'], 'Type': 'Call', 'OI': row['openInterest'],
                'GEX': gamma * row['openInterest'] * 100 * spot_price,
                'VEX': vanna * row['openInterest'] * 100 * spot_price
            })

        # Process Puts
        for _, row in opt_chain.puts.iterrows():
            if pd.isna(row['impliedVolatility']) or not row['openInterest']: continue
            gamma, vanna = black_scholes_greeks(spot_price, row['strike'], time_to_exp, risk_free_rate, row['impliedVolatility'])
            all_strikes_data.append({
                'Strike': row['strike'], 'Type': 'Put', 'OI': row['openInterest'],
                'GEX': -gamma * row['openInterest'] * 100 * spot_price,
                'VEX': -vanna * row['openInterest'] * 100 * spot_price
            })

    if not all_strikes_data:
        print("❌ Could not parse any options data records.")
        return

    df = pd.DataFrame(all_strikes_data)
    aggregated = df.groupby('Strike').agg({'GEX': 'sum', 'VEX': 'sum'}).reset_index()

    call_df = df[df['Type'] == 'Call']
    put_df = df[df['Type'] == 'Put']

    call_wall = call_df.loc[call_df['GEX'].idxmax()]['Strike'] if not call_df.empty else "N/A"
    put_wall = put_df.loc[put_df['GEX'].idxmin()]['Strike'] if not put_df.empty else "N/A"

    total_gex = aggregated['GEX'].sum()
    total_vex = aggregated['VEX'].sum()

    # Locate exact Gamma Flip boundary
    aggregated['GEX_Sign'] = np.sign(aggregated['GEX'])
    aggregated['Sign_Flip'] = aggregated['GEX_Sign'].diff().fillna(0)
    flip_candidates = aggregated[aggregated['Sign_Flip'] != 0]

    if not flip_candidates.empty:
        gamma_flip = flip_candidates.iloc[(flip_candidates['Strike'] - spot_price).abs().argmin()]['Strike']
    else:
        gamma_flip = "Undetermined"

    print("\n" + "="*40 + "\n🔥 LIVE HYBRID MARKET PROFILE ANALYSIS \n" + "="*40)
    print(f"🔹 Net System GEX:   ${total_gex:,.2f}")
    print(f"🔹 Net System VEX:   ${total_vex:,.2f}")
    print(f"🟩 Key Call Wall:    ${call_wall}")
    print(f"🟥 Key Put Wall:     ${put_wall}")
    print(f"🔀 Gamma Flip Level: ${gamma_flip}")
    print("="*40)

if __name__ == "__main__":
    # Can change to "QQQ" or "AAPL" smoothly
    fetch_and_calculate_gex("SPY")


⚡ Initializing Hybrid Data Stream for SPY...
🎯 Active Spot Price Locked: $764.29
📦 Pulling massive Options Open Interest profile via yfinance...

🔥 LIVE HYBRID MARKET PROFILE ANALYSIS 
🔹 Net System GEX:   $-450,957,290.29
🔹 Net System VEX:   $69,528,088,544.83
🟩 Key Call Wall:    $775.0
🟥 Key Put Wall:     $750.0
🔀 Gamma Flip Level: $764.0


In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Suppress operational pandas, matplotlib, and yfinance console warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

def get_tickers_from_user_csv(file_name="/content/sp500.csv"):
    """
    Extracts tickers safely from local working directories.
    Handles data cleansing formats (. to -) required for direct yfinance mapping.
    """
    print(f"[DEBUG 1] Checking system directory paths...")
    print(f"  > Current Working Directory: '{os.getcwd()}'")
    print(f"  > Looking for file target path: '{file_name}'")

    if not os.path.exists(file_name):
        if os.path.exists("./sp500.csv"):
            file_name = "./sp500.csv"
        elif os.path.exists("/sp500.csv"):
            file_name = "/sp500.csv"
        else:
            raise FileNotFoundError(f"CRITICAL ERROR: '{file_name}' was not detected in your folder list.")

    print(f"[DEBUG 3] File verified! Attempting to parse CSV layout data...")
    df_local = pd.read_csv(file_name)
    print(f"  > Successfully loaded CSV. Columns found in your file: {list(df_local.columns)}")

    if 'Symbol' not in df_local.columns:
        raise KeyError("Spreadsheet format error: Could not locate a column named 'Symbol'.")
    print(f"  > Utilizing column identifier: 'Symbol'")

    raw_tickers = df_local['Symbol'].dropna().astype(str).tolist()
    cleaned_tickers = [t.strip().replace('.', '-') for t in raw_tickers if t.strip()]
    cleaned_tickers = sorted(list(set(cleaned_tickers)))
    print(f"[DEBUG 4] Data processing successful. Found {len(raw_tickers)} raw symbols, cleaned down to {len(cleaned_tickers)} unique tickers.\n")
    return cleaned_tickers


def generate_volume_profile(df, lookback, bins=30):
    """
    Constructs an institutional Volume Profile matrix.
    Aggregates trading volumes over price intervals to isolate true POC and HVN nodes.
    """
    recent_data = df.iloc[-lookback:].copy()

    # Force absolute flattening to 1D arrays to prevent scalar conversion exceptions
    volumes = np.asarray(recent_data['Volume']).flatten()
    closes = np.asarray(recent_data['Close']).flatten()
    highs = np.asarray(recent_data['High']).flatten()
    lows = np.asarray(recent_data['Low']).flatten()

    price_min, price_max = lows.min(), highs.max()
    if price_max == price_min:
        return float(closes[-1]), float(closes[-1])

    price_bins = np.linspace(price_min, price_max, bins + 1)
    volume_profile = np.zeros(bins)

    for i in range(len(recent_data)):
        bar_range = highs[i] - lows[i]
        bar_vol = volumes[i]
        if bar_range == 0:
            bin_idx = np.clip(np.digitize(closes[i], price_bins) - 1, 0, bins - 1)
            volume_profile[bin_idx] += bar_vol
        else:
            for b in range(bins):
                bin_low = price_bins[b]
                bin_high = price_bins[b+1]
                overlap = min(highs[i], bin_high) - max(lows[i], bin_low)
                if overlap > 0:
                    volume_profile[b] += (overlap / bar_range) * bar_vol

    # POC Calculation
    poc_idx = np.argmax(volume_profile)
    target_poc = float((price_bins[poc_idx] + price_bins[poc_idx + 1]) / 2)

    # Secondary HVN Calculation via exclusion window masking
    mask_range = max(1, bins // 6)
    start_mask = max(0, poc_idx - mask_range)
    end_mask = min(bins, poc_idx + mask_range + 1)

    masked_profile = volume_profile.copy()
    masked_profile[start_mask:end_mask] = 0

    if masked_profile.max() > 0:
        hvn_idx = np.argmax(masked_profile)
        target_hvn = float((price_bins[hvn_idx] + price_bins[hvn_idx + 1]) / 2)
    else:
        sorted_indices = np.argsort(volume_profile)
        hvn_idx = sorted_indices[-2] if len(sorted_indices) > 1 else poc_idx
        target_hvn = float((price_bins[hvn_idx] + price_bins[hvn_idx + 1]) / 2)

    return round(target_poc, 2), round(target_hvn, 2)


def plot_vsa_profile_chart(df, ticker, timeframe, scan_type, target_poc, target_hvn, lookback, save_path):
    """
    Renders the live institutional Volume Profile technical chart panel.
    """
    plt.style.use('dark_background')
    fig, (ax_price, ax_vol) = plt.subplots(2, 1, figsize=(12, 7), gridspec_kw={'height_ratios': [2, 1]})

    fig.suptitle(f"{ticker} -- {timeframe.upper()} {scan_type.upper()} PROFILE ANALYSIS", fontsize=14, fontweight='bold', color='#00FFCC')

    plot_df = df.iloc[-(lookback + 15):].copy()
    x_indices = np.arange(len(plot_df))

    # Render price tracks safely
    ax_price.plot(x_indices, plot_df['Close'].values, color='#FFFFFF', linewidth=1.5, label='Closing Price', alpha=0.9)
    ax_price.axhline(target_poc, color='#FF3366', linestyle='--', linewidth=1.8, label=f'POC: ${target_poc:.2f}')
    ax_price.axhline(target_hvn, color='#FFCC00', linestyle=':', linewidth=1.8, label=f'HVN: ${target_hvn:.2f}')

    ax_price.set_ylabel('Price ($)', color='#FFFFFF', fontsize=11)
    ax_price.grid(True, linestyle=':', alpha=0.3, color='#555555')
    ax_price.legend(loc='upper left', framealpha=0.6, facecolor='#111111')

    # Sync visual volume histograms safely
    closes = plot_df['Close'].values
    opens = plot_df['Open'].values
    colors = ['#00FF66' if c >= o else '#FF3333' for c, o in zip(closes, opens)]

    ax_vol.bar(x_indices, plot_df['Volume'].values.flatten(), color=colors, alpha=0.75, width=0.7)
    ax_vol.set_ylabel('Volume', color='#FFFFFF', fontsize=11)
    ax_vol.grid(True, linestyle=':', alpha=0.3, color='#555555')
    ax_vol.set_xlim(ax_price.get_xlim())

    tick_step = max(1, len(plot_df) // 6)
    date_labels = [d.strftime('%Y-%m-%d') for d in plot_df.index]
    ax_vol.set_xticks(x_indices[::tick_step])
    ax_vol.set_xticklabels(date_labels[::tick_step], rotation=15, ha='right', fontsize=9)

    plt.tight_layout()
    plt.savefig(save_path, dpi=120, facecolor='#111111')
    plt.close()


def compute_vsa_metrics(df, lookback, idx, ticker, timeframe, scan_type, output_dir):
    """
    Applies multi-index column checks and maps structural market metrics securely.
    """
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    highs = np.asarray(df['High']).flatten()
    lows = np.asarray(df['Low']).flatten()
    closes = np.asarray(df['Close']).flatten()
    volumes = np.asarray(df['Volume']).flatten()

    bar_range = highs - lows
    bar_range_safe = np.where(bar_range == 0, 0.001, bar_range)
    buy_pressure_ratio = (closes - lows) / bar_range_safe
    buy_pressure_ratio = np.clip(buy_pressure_ratio, 0.0, 1.0)

    v_series = pd.Series(volumes, index=df.index)
    avg_volume = v_series.rolling(window=lookback).mean()

    current_volume = float(v_series.iloc[idx])
    current_avg_vol = float(avg_volume.iloc[idx])
    current_price = float(closes[idx])
    vol_ratio = current_volume / current_avg_vol

    if scan_type == "Distribution":
        dominance_pct = (1.0 - buy_pressure_ratio[idx]) * 100
    else:
        dominance_pct = buy_pressure_ratio[idx] * 100

    target_poc, target_hvn = generate_volume_profile(df, lookback)
    vsa_score = (vol_ratio * 50) + (dominance_pct * 1.5)

    img_name = f"{ticker}_{timeframe.capitalize()}_{scan_type.capitalize()}_profile.png"
    save_path = os.path.join(output_dir, img_name)
    plot_vsa_profile_chart(df, ticker, timeframe, scan_type, target_poc, target_hvn, lookback, save_path)

    return {
        "Current_Price": round(current_price, 2),
        "VSA_Score": round(vsa_score, 2),
        "Target_POC": round(target_poc, 2),
        "Target_HVN": round(target_hvn, 2),
        "Vol_vs_Avg": round(vol_ratio, 2),
        "Dominance_Pct": round(dominance_pct, 1),
        "Chart_Path": img_name
    }


In [ ]:
import datetime
from io import StringIO  # Imported to fix the read_html FutureWarning
import pandas as pd
import requests
import yfinance as yf
from tabulate import tabulate


def get_sp500_tickers():
    """Scrapes the live list of S&P 500 tickers directly from Wikipedia."""
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {"User-Agent": "Mozilla/5.0"}

    try:
        response = requests.get(url, headers=headers)
        # Fixed: Wrap response.text in StringIO to avoid pandas FutureWarning
        tables = pd.read_html(StringIO(response.text))
        sp500_df = tables[0]
        tickers = sp500_df["Symbol"].str.replace(".", "-", regex=False).tolist()
        print(f"Successfully loaded {len(tickers)} S&P 500 tickers dynamically.")
        return tickers
    except Exception as e:
        print(f"Error fetching live S&P 500 list: {e}. Falling back to default list.")
        return ["AAPL", "MSFT", "NVDA", "AMZN", "META", "GOOGL", "TSLA", "AVGO", "MRNA", "MU"]


def generate_thesis(symbol, option_type, strike, underlying_price, expiry_str, side, vol_oi_ratio):
    """Synthesizes qualitative thesis based on multi-variable option parameters."""
    today = datetime.date.today()
    expiry_date = datetime.datetime.strptime(expiry_str, "%Y-%m-%d").date()
    days_to_expiry = (expiry_date - today).days

    # 1. Expiry Horizon
    if days_to_expiry <= 14:
        horizon = "ultra short-dated"
    elif days_to_expiry <= 60:
        horizon = "medium-term"
    elif days_to_expiry <= 180:
        horizon = "intermediate"
    else:
        horizon = "long-dated LEAP"

    # 2. Moneyness
    pct_diff = (strike - underlying_price) / underlying_price if underlying_price > 0 else 0
    if abs(pct_diff) < 0.02:
        moneyness = "at-the-money"
    elif option_type == "CALL":
        moneyness = "out-of-the-money" if pct_diff > 0 else "in-the-money"
    else:
        moneyness = "out-of-the-money" if pct_diff < 0 else "in-the-money"

    # 3. Strategy Synthesis
    if option_type == "CALL":
        if "Ask" in side:
            if days_to_expiry <= 14:
                thesis = f"Aggressive {horizon} momentum sweep targeting immediate upside movement."
            elif moneyness == "out-of-the-money":
                thesis = f"High-conviction {horizon} bullish call accumulation targeting breakout past ${strike:.2f}."
            else:
                thesis = f"Directional {horizon} high-delta leveraged call position."
        elif "Bid" in side:
            thesis = f"Institutional covered call overwriting or profit-taking against existing long position."
        else:
            thesis = f"Large institutional block execution establishing structured upside exposure."
    else:  # PUT
        if "Ask" in side:
            if days_to_expiry <= 14:
                thesis = f"Tactical {horizon} protective hedge or downside speculation ahead of short-term catalysts."
            else:
                thesis = f"Institutional downside hedge or portfolio risk reduction against broader pullback risk."
        elif "Bid" in side:
            if days_to_expiry > 180:
                thesis = f"Ultra long-dated institutional LEAP put selling / cash-secured yield generation."
            else:
                thesis = f"Strategic put selling / spread financing to acquire shares at a lower cost basis."
        else:
            thesis = f"Multi-leg institutional protective collar or portfolio risk-reversal execution."

    return thesis


def fetch_unusual_options(tickers, min_volume=1000, min_vol_oi_ratio=2.0, min_dollar_val=100000, max_expiries=3):
    records = []
    total_scanned = 0

    for symbol in tickers:
        try:
            ticker = yf.Ticker(symbol)
            expirations = ticker.options

            if not expirations:
                continue

            fast_info = ticker.fast_info
            underlying_price = fast_info.get("lastPrice", 0) or fast_info.get("previousClose", 0)

            for expiry in expirations[:max_expiries]:
                chain = ticker.option_chain(expiry)

                for option_type, df in [("CALL", chain.calls), ("PUT", chain.puts)]:
                    if df.empty:
                        continue

                    for _, row in df.iterrows():
                        vol = row.get("volume", 0)
                        oi = row.get("openInterest", 0)

                        if pd.isna(vol) or vol < min_volume:
                            continue

                        adj_oi = oi if (not pd.isna(oi) and oi > 0) else 1
                        vol_oi_ratio = vol / adj_oi

                        if vol_oi_ratio >= min_vol_oi_ratio:
                            bid = row.get("bid", 0)
                            ask = row.get("ask", 0)
                            last_price = row.get("lastPrice", 0)

                            if bid > 0 and ask > 0:
                                avg_price = (bid + ask) / 2.0
                            else:
                                avg_price = last_price

                            total_dollar_val = vol * avg_price * 100

                            if total_dollar_val < min_dollar_val:
                                continue

                            if last_price >= ask and ask > 0:
                                side = "Ask Side"
                            elif last_price <= bid and bid > 0:
                                side = "Bid Side"
                            else:
                                side = "Mid / Block"

                            thesis = generate_thesis(
                                symbol=symbol,
                                option_type=option_type,
                                strike=row["strike"],
                                underlying_price=underlying_price,
                                expiry_str=expiry,
                                side=side,
                                vol_oi_ratio=vol_oi_ratio,
                            )

                            records.append(
                                {
                                    "TickerType": f"{symbol} {option_type}",
                                    "Strike Price": f"${row['strike']:.2f}",
                                    "Expiration Date": expiry,
                                    "Volume": int(vol),
                                    "Prior OI": int(oi) if pd.notna(oi) else 0,
                                    "Volume / OI": round(vol_oi_ratio, 2),
                                    "Average Price": f"${avg_price:.2f}",
                                    "Total Dollar Value": f"${total_dollar_val:,.2f}",
                                    "Execution Side": side,
                                    "Explanatory Thesis / Context": thesis,
                                }
                            )
            total_scanned += 1
            if total_scanned % 50 == 0:
                print(f"Scanned {total_scanned}/{len(tickers)} tickers...")

        except Exception as e:
            continue

    df_results = pd.DataFrame(records)
    if not df_results.empty:
        df_results = df_results.sort_values(by="Volume / OI", ascending=False).reset_index(drop=True)

    return df_results


if __name__ == "__main__":
    dynamic_watchlist = get_sp500_tickers()
    sample_scan = dynamic_watchlist[:50]

    print("\nScanning market for unusual options activity...\n")
    results = fetch_unusual_options(
        tickers=sample_scan,
        min_volume=1000,
        min_vol_oi_ratio=2.0,
        min_dollar_val=100000,
        max_expiries=3
    )

    if not results.empty:
        print(tabulate(results, headers="keys", tablefmt="github", showindex=False))
    else:
        print("No unusual options activity matching the criteria was found.")

Successfully loaded 503 S&P 500 tickers dynamically.

Scanning market for unusual options activity...

Scanned 50/50 tickers...
| TickerType   | Strike Price   | Expiration Date   |   Volume |   Prior OI |   Volume / OI | Average Price   | Total Dollar Value   | Execution Side   | Explanatory Thesis / Context                                                                       |
|--------------|----------------|-------------------|----------|------------|---------------|-----------------|----------------------|------------------|----------------------------------------------------------------------------------------------------|
| APH PUT      | $125.00        | 2026-09-18        |     1673 |          0 |       1673    | $45.85          | $7,670,705.00        | Bid Side         | Strategic put selling / spread financing to acquire shares at a lower cost basis.                  |
| APH PUT      | $110.00        | 2026-09-18        |     1070 |          0 |       1070    | $31.00       

In [ ]:
# modified pre market pull


In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from tradingview_screener import Query, Column


def fetch_prefiltered_universe(max_tickers: int = 500) -> list:
    """Uses TradingView API to fetch US optionable stocks trading below their 200 SMA."""
    print("Fetching pre-filtered universe from TradingView server-side screener...")

    # Server-side query execution
    q = (
        Query()
        .set_markets("america")
        .select("name", "close", "SMA200")
        .where(
            Column("type").isin(["stock", "dr"]),
            Column("subtype").isin(["common", "etf"]),
            Column("is_primary") == True,
            Column("close") < Column("SMA200"),  # Pre-filter: Below 200 MA
            Column("average_volume_30d_calc") > 500000,  # Liquidity filter
        )
        .order_by("market_cap_basic", ascending=False)
        .limit(max_tickers)
    )

    df_tv = q.get_scanner_data()[1]

    if df_tv.empty:
        print("No tickers returned from server-side screen.")
        return []

    tickers = df_tv["name"].tolist()
    print(f"Server pre-filter complete: {len(tickers)} stocks identified.")
    return tickers


def run_institutional_scanner(
    tickers: list,
    ma_type: str = "EMA",
    max_gap_pct: float = 8.0,
    consolidation_days: int = 21,
    max_consolidation_range: float = 12.0,
    breach_lookback: int = 5,
    min_rvol: float = 1.3,
    require_structure: bool = True,
) -> pd.DataFrame:
    """Runs detailed candle structure, consolidation, and breach logic on candidates."""
    results = []

    for ticker in tickers:
        try:
            tk = yf.Ticker(ticker)
            df = tk.history(period="1y", interval="1d")

            if len(df) < 200:
                continue

            # Calculate MAs
            if ma_type.upper() == "EMA":
                df["MA50"] = df["Close"].ewm(span=50, adjust=False).mean()
                df["MA200"] = df["Close"].ewm(span=200, adjust=False).mean()
            else:
                df["MA50"] = df["Close"].rolling(window=50).mean()
                df["MA200"] = df["Close"].rolling(window=200).mean()

            # Relative Volume (RVOL)
            df["Vol_SMA20"] = df["Volume"].rolling(window=20).mean()
            df["RVOL"] = np.where(
                df["Vol_SMA20"] > 0, df["Volume"] / df["Vol_SMA20"], 0
            )

            latest = df.iloc[-1]
            ma50_val = latest["MA50"]
            ma200_val = latest["MA200"]

            # Filter 1: 50 MA < 200 MA
            if ma50_val >= ma200_val:
                continue

            # Filter 2: Adjustable Gap %
            gap_pct = ((ma200_val - ma50_val) / ma200_val) * 100.0
            if gap_pct > max_gap_pct or gap_pct < 0:
                continue

            # Filter 3: Consolidation over 21 days (offset by lookback)
            cons_window = df.iloc[
                -(consolidation_days + breach_lookback) : -breach_lookback
            ]
            cons_high = cons_window["High"].max()
            cons_low = cons_window["Low"].min()
            cons_range_pct = ((cons_high - cons_low) / cons_low) * 100.0

            if cons_range_pct > max_consolidation_range:
                continue

            # Filter 4: Breach within last 5 days
            recent_window = df.iloc[-breach_lookback:]
            breach_mask = recent_window["High"] >= recent_window["MA50"]

            if not breach_mask.any():
                continue

            # Filter 5: Momentum structure (HH / HL + RVOL)
            breach_indices = recent_window[breach_mask].index
            momentum_confirmed = False
            best_rvol = 0.0

            for idx in breach_indices:
                loc = df.index.get_loc(idx)
                if loc < 1:
                    continue

                curr_bar = df.iloc[loc]
                prev_bar = df.iloc[loc - 1]

                hh_hl = (curr_bar["High"] > prev_bar["High"]) and (
                    curr_bar["Low"] > prev_bar["Low"]
                )
                rvol_pass = curr_bar["RVOL"] >= min_rvol

                if (not require_structure or hh_hl) and rvol_pass:
                    momentum_confirmed = True
                    best_rvol = max(best_rvol, round(curr_bar["RVOL"], 2))
                    break

            if not momentum_confirmed:
                continue

            results.append(
                {
                    "Ticker": ticker,
                    "Close": round(latest["Close"], 2),
                    "MA50": round(ma50_val, 2),
                    "MA200": round(ma200_val, 2),
                    "Gap_Pct": round(gap_pct, 2),
                    "Cons_Range_Pct": round(cons_range_pct, 2),
                    "Breach_RVOL": best_rvol,
                }
            )

        except Exception:
            continue

    return pd.DataFrame(results)


if __name__ == "__main__":
    # Step 1: Pre-filter universe dynamically
    candidate_tickers = fetch_prefiltered_universe(max_tickers=300)

    # Step 2: Run strategy scanner
    if candidate_tickers:
        df_results = run_institutional_scanner(
            tickers=candidate_tickers,
            ma_type="EMA",
           # max_gap_pct=8.0,
           # consolidation_days=21,
          #  max_consolidation_range=12.0,
            breach_lookback=5,
            min_rvol=1.3,
            require_structure=True,
        )

        print("\n=== FINAL MATCHES ===")
        print(df_results)

Fetching pre-filtered universe from TradingView server-side screener...
Server pre-filter complete: 300 stocks identified.

=== FINAL MATCHES ===
  Ticker   Close    MA50   MA200  Gap_Pct  Cons_Range_Pct  Breach_RVOL
0     PG  146.60  145.84  146.56     0.49            9.61         1.53
1    SUI  123.43  121.60  123.14     1.25            8.82         6.59
2    CPB   24.16   22.47   23.77     5.46           10.15         2.24


In [ ]:
#extracing tickers which are below 200 and 50 EMA but in last 5 days candle crossed 50 ema.

import pandas as pd
import numpy as np
import yfinance as yf
from tradingview_screener import Query, Column
from typing import List


def fetch_prefiltered_universe(max_tickers: int = 300) -> List[str]:
    """Fetches liquid US stocks trading below their 200 SMA from TradingView."""
    print("Fetching pre-filtered universe from TradingView...")

    q = (
        Query()
        .set_markets("america")
        .select("name", "close", "SMA200")
        .where(
            Column("type").isin(["stock", "dr"]),
            Column("subtype").isin(["common", "etf"]),
            Column("is_primary") == True,
            Column("close") < Column("SMA200"),
            Column("average_volume_30d_calc") > 500000,
        )
        .order_by("market_cap_basic", ascending=False)
        .limit(max_tickers)
    )

    df_tv = q.get_scanner_data()[1]
    if df_tv.empty:
        return []
    return df_tv["name"].tolist()


def run_unfiltered_metric_scanner(
    tickers: List[str],
    ma_type: str = "EMA",
    consolidation_days: int = 21,
    breach_lookback: int = 5,
) -> pd.DataFrame:
    """
    Calculates Gap %, Consolidation Range, and RVOL for transparency,
    but ONLY filters on MA alignment, 5-day 50 MA breach, and HH/HL structure.
    """
    results = []

    for ticker in tickers:
        try:
            tk = yf.Ticker(ticker)
            df = tk.history(period="1y", interval="1d")

            if len(df) < 200:
                continue

            # 1. Moving Average Calculations
            if ma_type.upper() == "EMA":
                df["MA50"] = df["Close"].ewm(span=50, adjust=False).mean()
                df["MA200"] = df["Close"].ewm(span=200, adjust=False).mean()
            else:
                df["MA50"] = df["Close"].rolling(window=50).mean()
                df["MA200"] = df["Close"].rolling(window=200).mean()

            # 2. Volume & RVOL Calculation
            df["Vol_SMA20"] = df["Volume"].rolling(window=20).mean()
            df["RVOL"] = np.where(
                df["Vol_SMA20"] > 0, df["Volume"] / df["Vol_SMA20"], 0
            )

            latest = df.iloc[-1]
            ma50_val = latest["MA50"]
            ma200_val = latest["MA200"]

            # ACTIVE FILTER 1: Fast MA below Slow MA
            if ma50_val >= ma200_val:
                continue

            # INFORMATIONAL CALCULATIONS (NO FILTERING APPLIED)
            # Gap Percentage
            gap_pct = ((ma200_val - ma50_val) / ma200_val) * 100.0

            # Consolidation Range Percentage (prior 21 trading days)
            cons_window = df.iloc[
                -(consolidation_days + breach_lookback) : -breach_lookback
            ]
            cons_high = cons_window["High"].max()
            cons_low = cons_window["Low"].min()
            cons_range_pct = ((cons_high - cons_low) / cons_low) * 100.0

            # ACTIVE FILTER 2: 50 MA Breach within the last 5 days
            recent_window = df.iloc[-breach_lookback:]
            breach_mask = recent_window["High"] >= recent_window["MA50"]

            if not breach_mask.any():
                continue

            # ACTIVE FILTER 3: Higher High + Higher Low structure on breach day
            breach_indices = recent_window[breach_mask].index
            momentum_confirmed = False
            breach_rvol = 0.0

            for idx in breach_indices:
                loc = df.index.get_loc(idx)
                if loc < 1:
                    continue

                curr_bar = df.iloc[loc]
                prev_bar = df.iloc[loc - 1]

                # Check structural HH/HL on breach bar
                hh_hl = (curr_bar["High"] > prev_bar["High"]) and (
                    curr_bar["Low"] > prev_bar["Low"]
                )

                if hh_hl:
                    momentum_confirmed = True
                    breach_rvol = round(curr_bar["RVOL"], 2)
                    break

            if not momentum_confirmed:
                continue

            # Stock passes core structure; append all metrics for display
            results.append(
                {
                    "Ticker": ticker,
                    "Close": round(latest["Close"], 2),
                    "MA50": round(ma50_val, 2),
                    "MA200": round(ma200_val, 2),
                    "Gap_Pct_Calc": round(gap_pct, 2),
                    "Cons_Range_Pct_Calc": round(cons_range_pct, 2),
                    "Breach_RVOL_Calc": breach_rvol,
                }
            )

        except Exception:
            continue

    return pd.DataFrame(results)


if __name__ == "__main__":
    universe = fetch_prefiltered_universe(max_tickers=300)

    if universe:
        df_out = run_unfiltered_metric_scanner(
            tickers=universe,
            ma_type="EMA",
            consolidation_days=21,
            breach_lookback=5,
        )

        print(f"\nScan Complete. Total Hits: {len(df_out)}")
        print(df_out.to_string(index=False))

Fetching pre-filtered universe from TradingView...

Scan Complete. Total Hits: 114
Ticker  Close   MA50  MA200  Gap_Pct_Calc  Cons_Range_Pct_Calc  Breach_RVOL_Calc
  TSLA 348.95 358.40 385.79          7.10                30.01              1.69
   WMT 106.49 113.52 115.03          1.31                 9.05              0.97
    PG 146.60 145.84 146.56          0.49                 9.61              1.53
    HD 337.43 338.12 346.02          2.29                10.93              1.24
  NFLX  80.01  76.86  88.65         13.30                18.05              1.01
   PEP 144.67 141.13 145.07          2.72                 8.75              0.88
  TMUS 182.62 182.48 196.66          7.21                15.56              0.94
   MCD 272.54 273.37 289.12          5.45                 6.80              0.91
  ISRG 373.57 390.94 441.43         11.44                23.26              0.73
   SYK 327.81 329.19 338.23          2.67                14.86              0.77
   LOW 216.98 216.57 230.5

In [ ]:
import datetime
from io import StringIO  # Imported to fix the read_html FutureWarning
import pandas as pd
import requests
import yfinance as yf
from tabulate import tabulate


def get_sp500_tickers():
    """Scrapes the live list of S&P 500 tickers directly from Wikipedia."""
    url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
    headers = {"User-Agent": "Mozilla/5.0"}

    try:
        response = requests.get(url, headers=headers)
        # Fixed: Wrap response.text in StringIO to avoid pandas FutureWarning
        tables = pd.read_html(StringIO(response.text))
        sp500_df = tables[0]
        tickers = sp500_df["Symbol"].str.replace(".", "-", regex=False).tolist()
        print(f"Successfully loaded {len(tickers)} S&P 500 tickers dynamically.")
        return tickers
    except Exception as e:
        print(f"Error fetching live S&P 500 list: {e}. Falling back to default list.")
        return ["AAPL", "MSFT", "NVDA", "AMZN", "META", "GOOGL", "TSLA", "AVGO", "MRNA", "MU","QQQ","SPY","spxw","GOOG","ORCL"]


def generate_thesis(symbol, option_type, strike, underlying_price, expiry_str, side, vol_oi_ratio):
    """Synthesizes qualitative thesis based on multi-variable option parameters."""
    today = datetime.date.today()
    expiry_date = datetime.datetime.strptime(expiry_str, "%Y-%m-%d").date()
    days_to_expiry = (expiry_date - today).days

    # 1. Expiry Horizon
    if days_to_expiry <= 14:
        horizon = "ultra short-dated"
    elif days_to_expiry <= 60:
        horizon = "medium-term"
    elif days_to_expiry <= 180:
        horizon = "intermediate"
    else:
        horizon = "long-dated LEAP"

    # 2. Moneyness
    pct_diff = (strike - underlying_price) / underlying_price if underlying_price > 0 else 0
    if abs(pct_diff) < 0.02:
        moneyness = "at-the-money"
    elif option_type == "CALL":
        moneyness = "out-of-the-money" if pct_diff > 0 else "in-the-money"
    else:
        moneyness = "out-of-the-money" if pct_diff < 0 else "in-the-money"

    # 3. Strategy Synthesis
    if option_type == "CALL":
        if "Ask" in side:
            if days_to_expiry <= 14:
                thesis = f"Aggressive {horizon} momentum sweep targeting immediate upside movement."
            elif moneyness == "out-of-the-money":
                thesis = f"High-conviction {horizon} bullish call accumulation targeting breakout past ${strike:.2f}."
            else:
                thesis = f"Directional {horizon} high-delta leveraged call position."
        elif "Bid" in side:
            thesis = f"Institutional covered call overwriting or profit-taking against existing long position."
        else:
            thesis = f"Large institutional block execution establishing structured upside exposure."
    else:  # PUT
        if "Ask" in side:
            if days_to_expiry <= 14:
                thesis = f"Tactical {horizon} protective hedge or downside speculation ahead of short-term catalysts."
            else:
                thesis = f"Institutional downside hedge or portfolio risk reduction against broader pullback risk."
        elif "Bid" in side:
            if days_to_expiry > 180:
                thesis = f"Ultra long-dated institutional LEAP put selling / cash-secured yield generation."
            else:
                thesis = f"Strategic put selling / spread financing to acquire shares at a lower cost basis."
        else:
            thesis = f"Multi-leg institutional protective collar or portfolio risk-reversal execution."

    return thesis


def fetch_unusual_options(tickers, min_volume=1000, min_vol_oi_ratio=2.0, min_dollar_val=100000, max_expiries=3):
    records = []
    total_scanned = 0

    for symbol in tickers:
        try:
            ticker = yf.Ticker(symbol)
            expirations = ticker.options

            if not expirations:
                continue

            fast_info = ticker.fast_info
            underlying_price = fast_info.get("lastPrice", 0) or fast_info.get("previousClose", 0)

            for expiry in expirations[:max_expiries]:
                chain = ticker.option_chain(expiry)

                for option_type, df in [("CALL", chain.calls), ("PUT", chain.puts)]:
                    if df.empty:
                        continue

                    for _, row in df.iterrows():
                        vol = row.get("volume", 0)
                        oi = row.get("openInterest", 0)

                        if pd.isna(vol) or vol < min_volume:
                            continue

                        adj_oi = oi if (not pd.isna(oi) and oi > 0) else 1
                        vol_oi_ratio = vol / adj_oi

                        if vol_oi_ratio >= min_vol_oi_ratio:
                            bid = row.get("bid", 0)
                            ask = row.get("ask", 0)
                            last_price = row.get("lastPrice", 0)

                            if bid > 0 and ask > 0:
                                avg_price = (bid + ask) / 2.0
                            else:
                                avg_price = last_price

                            total_dollar_val = vol * avg_price * 100

                            if total_dollar_val < min_dollar_val:
                                continue

                            if last_price >= ask and ask > 0:
                                side = "Ask Side"
                            elif last_price <= bid and bid > 0:
                                side = "Bid Side"
                            else:
                                side = "Mid / Block"

                            thesis = generate_thesis(
                                symbol=symbol,
                                option_type=option_type,
                                strike=row["strike"],
                                underlying_price=underlying_price,
                                expiry_str=expiry,
                                side=side,
                                vol_oi_ratio=vol_oi_ratio,
                            )

                            records.append(
                                {
                                    "TickerType": f"{symbol} {option_type}",
                                    "Strike Price": f"${row['strike']:.2f}",
                                    "Expiration Date": expiry,
                                    "Volume": int(vol),
                                    "Prior OI": int(oi) if pd.notna(oi) else 0,
                                    "Volume / OI": round(vol_oi_ratio, 2),
                                    "Average Price": f"${avg_price:.2f}",
                                    "Total Dollar Value": f"${total_dollar_val:,.2f}",
                                    "Execution Side": side,
                                    "Explanatory Thesis / Context": thesis,
                                }
                            )
            total_scanned += 1
            if total_scanned % 50 == 0:
                print(f"Scanned {total_scanned}/{len(tickers)} tickers...")

        except Exception as e:
            continue

    df_results = pd.DataFrame(records)
    if not df_results.empty:
        df_results = df_results.sort_values(by="Volume / OI", ascending=False).reset_index(drop=True)

    return df_results


if __name__ == "__main__":
    dynamic_watchlist = get_sp500_tickers()
    sample_scan = dynamic_watchlist[:50]

    print("\nScanning market for unusual options activity...\n")
    results = fetch_unusual_options(
        tickers=sample_scan,
        min_volume=1000,
        min_vol_oi_ratio=2.0,
        min_dollar_val=100000,
        max_expiries=3
    )

    if not results.empty:
        print(tabulate(results, headers="keys", tablefmt="github", showindex=False))
    else:
        print("No unusual options activity matching the criteria was found.")

Successfully loaded 503 S&P 500 tickers dynamically.

Scanning market for unusual options activity...

Scanned 50/50 tickers...
| TickerType   | Strike Price   | Expiration Date   |   Volume |   Prior OI |   Volume / OI | Average Price   | Total Dollar Value   | Execution Side   | Explanatory Thesis / Context                                                                       |
|--------------|----------------|-------------------|----------|------------|---------------|-----------------|----------------------|------------------|----------------------------------------------------------------------------------------------------|
| AMZN PUT     | $267.50        | 2026-09-21        |     2872 |          0 |       2872    | $13.95          | $4,006,440.00        | Mid / Block      | Multi-leg institutional protective collar or portfolio risk-reversal execution.                    |
| AMZN PUT     | $265.00        | 2026-09-21        |     4311 |          2 |       2155.5  | $11.80       

In [ ]:
import datetime
import os

from io import StringIO
import pandas as pd
import requests
import yfinance as yf
from tabulate import tabulate


def get_tickers_from_file(file_path="/content/sp500.csv"):
    """Reads tickers directly from a CSV file located in the target directory.

    Falls back to current directory or a default list if the file is not found.
    """
    # Check primary path (/content/sp500.csv) or fallback to local directory
    target_path = file_path
    if not os.path.exists(target_path) and os.path.exists("sp500.csv"):
        target_path = "sp500.csv"

    try:
        df = pd.read_csv(target_path)

        # Automatically locate the ticker symbol column
        symbol_col = None
        for col in ["Symbol", "symbol", "Ticker", "ticker"]:
            if col in df.columns:
                symbol_col = col
                break

        if symbol_col is None:
            # Fall back to the first column if no explicit header matches
            symbol_col = df.columns[0]

        # Convert dot notation (BRK.B -> BRK-B) for yfinance compatibility
        tickers = (
            df[symbol_col].dropna().astype(str).str.replace(".", "-", regex=False).tolist()
        )
        print(
            f"Successfully loaded {len(tickers)} tickers from '{target_path}'."
        )
        return tickers

    except Exception as e:
        print(
            f"Error loading ticker file from '{target_path}': {e}. Falling back to default list."
        )
        return [
            "AAPL",
            "MSFT",
            "NVDA",
            "AMZN",
            "META",
            "GOOGL",
            "TSLA",
            "AVGO",
            "MRNA",
            "MU",
        ]


def generate_thesis(
    symbol,
    option_type,
    strike,
    underlying_price,
    expiry_str,
    side,
    vol_oi_ratio,
):
    """Synthesizes qualitative thesis based on multi-variable option parameters."""
    today = datetime.date.today()
    expiry_date = datetime.datetime.strptime(expiry_str, "%Y-%m-%d").date()
    days_to_expiry = (expiry_date - today).days

    # 1. Expiry Horizon
    if days_to_expiry <= 14:
        horizon = "ultra short-dated"
    elif days_to_expiry <= 60:
        horizon = "medium-term"
    elif days_to_expiry <= 180:
        horizon = "intermediate"
    else:
        horizon = "long-dated LEAP"

    # 2. Moneyness
    pct_diff = (
        (strike - underlying_price) / underlying_price
        if underlying_price > 0
        else 0
    )
    if abs(pct_diff) < 0.02:
        moneyness = "at-the-money"
    elif option_type == "CALL":
        moneyness = "out-of-the-money" if pct_diff > 0 else "in-the-money"
    else:
        moneyness = "out-of-the-money" if pct_diff < 0 else "in-the-money"

    # 3. Strategy Synthesis
    if option_type == "CALL":
        if "Ask" in side:
            if days_to_expiry <= 14:
                thesis = f"Aggressive {horizon} momentum sweep targeting immediate upside movement."
            elif moneyness == "out-of-the-money":
                thesis = f"High-conviction {horizon} bullish call accumulation targeting breakout past ${strike:.2f}."
            else:
                thesis = f"Directional {horizon} high-delta leveraged call position."
        elif "Bid" in side:
            thesis = f"Institutional covered call overwriting or profit-taking against existing long position."
        else:
            thesis = f"Large institutional block execution establishing structured upside exposure."
    else:  # PUT
        if "Ask" in side:
            if days_to_expiry <= 14:
                thesis = f"Tactical {horizon} protective hedge or downside speculation ahead of short-term catalysts."
            else:
                thesis = f"Institutional downside hedge or portfolio risk reduction against broader pullback risk."
        elif "Bid" in side:
            if days_to_expiry > 180:
                thesis = f"Ultra long-dated institutional LEAP put selling / cash-secured yield generation."
            else:
                thesis = f"Strategic put selling / spread financing to acquire shares at a lower cost basis."
        else:
            thesis = f"Multi-leg institutional protective collar or portfolio risk-reversal execution."

    return thesis


def fetch_unusual_options(
    tickers,
    min_volume=1000,
    min_vol_oi_ratio=2.0,
    min_dollar_val=100000,
    max_expiries=3,
):
    records = []
    total_scanned = 0

    for symbol in tickers:
        try:
            ticker = yf.Ticker(symbol)
            expirations = ticker.options

            if not expirations:
                continue

            fast_info = ticker.fast_info
            underlying_price = fast_info.get(
                "lastPrice", 0
            ) or fast_info.get("previousClose", 0)

            for expiry in expirations[:max_expiries]:
                chain = ticker.option_chain(expiry)

                for option_type, df in [
                    ("CALL", chain.calls),
                    ("PUT", chain.puts),
                ]:
                    if df.empty:
                        continue

                    for _, row in df.iterrows():
                        vol = row.get("volume", 0)
                        oi = row.get("openInterest", 0)

                        if pd.isna(vol) or vol < min_volume:
                            continue

                        adj_oi = oi if (not pd.isna(oi) and oi > 0) else 1
                        vol_oi_ratio = vol / adj_oi

                        if vol_oi_ratio >= min_vol_oi_ratio:
                            bid = row.get("bid", 0)
                            ask = row.get("ask", 0)
                            last_price = row.get("lastPrice", 0)

                            if bid > 0 and ask > 0:
                                avg_price = (bid + ask) / 2.0
                            else:
                                avg_price = last_price

                            total_dollar_val = vol * avg_price * 100

                            if total_dollar_val < min_dollar_val:
                                continue

                            if last_price >= ask and ask > 0:
                                side = "Ask Side"
                            elif last_price <= bid and bid > 0:
                                side = "Bid Side"
                            else:
                                side = "Mid / Block"

                            thesis = generate_thesis(
                                symbol=symbol,
                                option_type=option_type,
                                strike=row["strike"],
                                underlying_price=underlying_price,
                                expiry_str=expiry,
                                side=side,
                                vol_oi_ratio=vol_oi_ratio,
                            )

                            records.append(
                                {
                                    "TickerType": f"{symbol} {option_type}",
                                    "Strike Price": f"${row['strike']:.2f}",
                                    "Expiration Date": expiry,
                                    "Volume": int(vol),
                                    "Prior OI": int(oi) if pd.notna(oi) else 0,
                                    "Volume / OI": round(vol_oi_ratio, 2),
                                    "Average Price": f"${avg_price:.2f}",
                                    "Total Dollar Value": f"${total_dollar_val:,.2f}",
                                    "Execution Side": side,
                                    "Explanatory Thesis / Context": thesis,
                                }
                            )
            total_scanned += 1
            if total_scanned % 50 == 0:
                print(f"Scanned {total_scanned}/{len(tickers)} tickers...")

        except Exception as e:
            continue

    df_results = pd.DataFrame(records)
    if not df_results.empty:
        df_results = df_results.sort_values(
            by="Volume / OI", ascending=False
        ).reset_index(drop=True)

    return df_results


if __name__ == "__main__":
    # Load tickers dynamically from /content/sp500.csv
    dynamic_watchlist = get_tickers_from_file("/content/sp500.csv")

    # Limit scan count if testing speed (e.g. dynamic_watchlist[:50]), or scan full universe
  # sample_scan = dynamic_watchlist[:50]

    print("\nScanning market for unusual options activity...\n")
    results = fetch_unusual_options(
        tickers=dynamic_watchlist,
        min_volume=1000,
        min_vol_oi_ratio=2.0,
        min_dollar_val=100000,
        max_expiries=3,
    )

    if not results.empty:
        print(
            tabulate(
                results, headers="keys", tablefmt="github", showindex=False
            )
        )
    else:
        print("No unusual options activity matching the criteria was found.")

Successfully loaded 518 tickers from '/content/sp500.csv'.

Scanning market for unusual options activity...

Scanned 50/518 tickers...
Scanned 100/518 tickers...
Scanned 150/518 tickers...
Scanned 200/518 tickers...
Scanned 250/518 tickers...
Scanned 300/518 tickers...
Scanned 350/518 tickers...
Scanned 400/518 tickers...
Scanned 450/518 tickers...
Scanned 500/518 tickers...
| TickerType   | Strike Price   | Expiration Date   |   Volume |   Prior OI |   Volume / OI | Average Price   | Total Dollar Value   | Execution Side   | Explanatory Thesis / Context                                                                       |
|--------------|----------------|-------------------|----------|------------|---------------|-----------------|----------------------|------------------|----------------------------------------------------------------------------------------------------|
| QQQ PUT      | $734.00        | 2026-09-22        |    30210 |          0 |      30210    | $0.57           | 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta

DEFAULT_DESK_WATCHLIST = ["SPY", "QQQ", "AAPL", "MSFT", "NVDA", "AMZN", "GOOGL", "META", "TSLA"]

def load_tickers_from_csv():
    potential_paths = ["content/sp500.csv", "./content/sp500.csv", "sp500.csv", "./sp500.csv"]
    selected_path = None
    for path in potential_paths:
        if os.path.exists(path):
            selected_path = path
            break
    if selected_path is None:
        print("⚠️ [PATH NOTICE]: File 'sp500.csv' not detected inside current directory container.")
        print(f"🚀 [FALLBACK]: Initializing default desk tracking matrix: {DEFAULT_DESK_WATCHLIST}\n")
        return DEFAULT_DESK_WATCHLIST
    try:
        print(f"✅ [SUCCESS]: Found source dataset at path: '{selected_path}'")
        df = pd.read_csv(selected_path)
        df.columns = [c.strip() for c in df.columns]
        symbol_col = None
        for col in ['Symbol', 'Ticker', 'symbol', 'ticker']:
            if col in df.columns:
                symbol_col = col
                break
        if symbol_col is None:
            print(f"⚠️ [FORMAT WARNING]: Missing a dedicated 'Symbol' column header in file rows.")
            return DEFAULT_DESK_WATCHLIST
        tickers = df[symbol_col].dropna().astype(str).str.strip().unique().tolist()
        tickers = [t for t in tickers if t and not t.lower() in ['symbol', 'test', 'ticker', 'nan']]
        return tickers if tickers else DEFAULT_DESK_WATCHLIST
    except Exception as e:
        print(f"💥 [CRASH WARNING]: Failed parsing file metadata block: {e}")
        return DEFAULT_DESK_WATCHLIST

def calculate_vsa_and_indicators(daily_df, weekly_df, monthly_df):
    if len(daily_df) < 30: return None
    df = daily_df.copy()
    df['bar_range'] = df['High'] - df['Low']
    df['buy_pressure_ratio'] = np.where(df['bar_range'] > 0, (df['Close'] - df['Low']) / df['bar_range'], 0.5)
    df['BuyVolume'] = df['Volume'] * df['buy_pressure_ratio']
    df['SellVolume'] = df['Volume'] * (1 - df['buy_pressure_ratio'])
    df['absSellVol'] = df['SellVolume'].abs()

    df['absSellVol_1'] = df['absSellVol'].shift(1)
    df['absSellVol_2'] = df['absSellVol'].shift(2)
    df['BuyVolume_1'] = df['BuyVolume'].shift(1)
    df['BuyVolume_2'] = df['BuyVolume'].shift(2)

    drop_mod = 1 - (15.0 / 100.0)
    df['sellersDryingUp'] = (df['absSellVol'] < df['absSellVol_1'] * drop_mod) & (df['absSellVol_1'] < df['absSellVol_2'] * drop_mod)
    df['buyersHolding'] = (df['BuyVolume'] >= df['BuyVolume_1'] * 0.90) & (df['BuyVolume_1'] >= df['BuyVolume_2'] * 0.90)
    df['buyersDryingUp'] = (df['BuyVolume'] < df['BuyVolume_1'] * drop_mod) & (df['BuyVolume_1'] < df['BuyVolume_2'] * drop_mod)
    df['sellersHolding'] = (df['absSellVol'] >= df['absSellVol_1'] * 0.90) & (df['absSellVol_1'] >= df['absSellVol_2'] * drop_mod)

    df['isSupplyExhaustion'] = df['sellersDryingUp'] & df['buyersHolding']
    df['isDemandExhaustion'] = df['buyersDryingUp'] & df['sellersHolding']

    high_low = df['High'] - df['Low']
    high_close = (df['High'] - df['Close'].shift(1)).abs()
    low_close = (df['Low'] - df['Close'].shift(1)).abs()
    ranges = pd.concat([high_low, high_close, low_close], axis=1)
    df['ATR'] = ranges.max(axis=1).ewm(alpha=1/14, adjust=False).mean()
    df['isATRExpansion'] = df['bar_range'] > (df['ATR'] * 1.3)
    df['closeLocationRatio'] = np.where(df['bar_range'] > 0, (df['Close'] - df['Low']) / df['bar_range'], 0.5)
    df['vol_avg20'] = df['Volume'].rolling(window=20).mean()
    df['isHighVolume'] = df['Volume'] >= (df['vol_avg20'] * 1.3)

    df['buyClimax_Fakeout'] = (df['BuyVolume'] >= df['Volume'] * 0.60) & df['isHighVolume'] & df['isATRExpansion'] & (df['closeLocationRatio'] >= 0.75)
    df['sellClimax_Fakeout'] = (df['absSellVol'] >= df['Volume'] * 0.60) & df['isHighVolume'] & df['isATRExpansion'] & (df['closeLocationRatio'] <= 0.25)
    df['standardBuyClimax'] = (df['BuyVolume'] >= df['Volume'] * 0.75) & df['isHighVolume']
    df['standardSellClimax'] = (df['absSellVol'] >= df['Volume'] * 0.75) & df['isHighVolume']

    df['Yellow_Buy_Dot'] = df['standardBuyClimax'] & ~df['buyClimax_Fakeout']
    df['Yellow_Sell_Dot'] = df['standardSellClimax'] & ~df['sellClimax_Fakeout']
    df['Orange_Buy_Dot'] = df['buyClimax_Fakeout']
    df['Orange_Sell_Dot'] = df['sellClimax_Fakeout']

    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    df['RSI'] = 100 - (100 / (1 + (gain / (loss + 1e-10))))

    mf_mul = np.where(df['bar_range'] > 0, ((df['Close'] - df['Low']) - (df['High'] - df['Close'])) / df['bar_range'], 0.0)
    df['CMF'] = (mf_mul * df['Volume']).rolling(window=20).sum() / (df['Volume'].rolling(window=20).sum() + 1e-10)

    recent_w_vol = weekly_df['Volume'].iloc[-1] if len(weekly_df) > 0 else 0
    w_vol_avg = weekly_df['Volume'].rolling(20).mean().iloc[-1] if len(weekly_df) > 20 else 1
    recent_m_vol = monthly_df['Volume'].iloc[-1] if len(monthly_df) > 0 else 0
    m_vol_avg = monthly_df['Volume'].rolling(12).mean().iloc[-1] if len(monthly_df) > 12 else 1
    df['Weekly_Vol_Status'] = "Elevated" if recent_w_vol > w_vol_avg * 1.2 else "Normal"
    df['Monthly_Vol_Status'] = "Elevated" if recent_m_vol > m_vol_avg * 1.2 else "Normal"
    return df

def backtest_success_rates(df):
    results = {'Yellow_Buy': {'rate': 0.0}, 'Yellow_Sell': {'rate': 0.0}, 'Orange_Buy': {'rate': 0.0}, 'Orange_Sell': {'rate': 0.0}}
    next_day_bullish = df['Close'].shift(-1) > df['Close']
    next_day_bearish = df['Close'].shift(-1) < df['Close']
    for key, col in [('Yellow_Buy', 'Yellow_Buy_Dot'), ('Yellow_Sell', 'Yellow_Sell_Dot'), ('Orange_Buy', 'Orange_Buy_Dot'), ('Orange_Sell', 'Orange_Sell_Dot')]:
        triggers = df[df[col] == True]
        if len(triggers) > 0:
            successes = len(df[(df[col] == True) & (next_day_bullish if 'Buy' in key else next_day_bearish)])
            results[key]['rate'] = round((successes / len(triggers)) * 100, 1)
    return results

def run_screener():
    tickers = load_tickers_from_csv()
    print(f"🔍 Initializing matrix scan across {len(tickers)} tracking targets...")
    dashboard_records = []
    end_date = datetime.now()
    start_date = end_date - timedelta(days=730)
    for ticker in tickers:
        try:
            d_data = yf.download(ticker, start=start_date, end=end_date, progress=False)
            w_data = yf.download(ticker, start=start_date, end=end_date, interval="1wk", progress=False)
            m_data = yf.download(ticker, start=start_date, end=end_date, interval="1mo", progress=False)
            if d_data.empty: continue
            df = calculate_vsa_and_indicators(d_data, w_data, m_data)
            if df is None or df.empty: continue
            rates = backtest_success_rates(df)
            last_bar = df.iloc[-1]
            active_color, thesis, bias, profile, success_pct = "None", "No Pattern", "Neutral", "Balance", 0.0
            if last_bar['Yellow_Buy_Dot']:
                active_color, thesis, bias, profile, success_pct = "Yellow", "Exhaustion Climax Reversal", "Bullish", "Accumulation", rates['Yellow_Buy']['rate']
            elif last_bar['Yellow_Sell_Dot']:
                active_color, thesis, bias, profile, success_pct = "Yellow", "Exhaustion Climax Reversal", "Bearish", "Distribution", rates['Yellow_Sell']['rate']
            elif last_bar['Orange_Buy_Dot']:
                active_color, thesis, bias, profile, success_pct = "Orange", "Institutional Absorption Momentum", "Bullish", "Aggressive Accumulation", rates['Orange_Buy']['rate']
            elif last_bar['Orange_Sell_Dot']:
                active_color, thesis, bias, profile, success_pct = "Orange", "Institutional Liquidation Absorption", "Bearish", "Aggressive Distribution", rates['Orange_Sell']['rate']
            if active_color != "None":
                dashboard_records.append({'Ticker': ticker, 'Dot Color': active_color, 'Thesis': thesis, 'Directional Bias': bias, 'Auction Profile': profile, 'Weekly Vol': last_bar['Weekly_Vol_Status'], 'Monthly Vol': last_bar['Monthly_Vol_Status'], 'Backtested Success %': f"{success_pct}%", 'RSI (14)': round(float(last_bar['RSI']), 1), 'CMF (20)': round(float(last_bar['CMF']), 2)})
        except: continue
    output_df = pd.DataFrame(dashboard_records)
    print("\n" + "="*75 + "\n🔥 LIVE INSTITUTIONAL CTA TRADING DASHBOARD\n" + "="*75)
    if output_df.empty:
        print("No tickers currently registering an active Yellow or Orange dot setup today.")
    else:
        print(output_df.to_string(index=False))
        if not os.path.exists("content"): os.makedirs("content")
        output_df.to_csv("content/cta_scan_results.csv", index=False)
        print("\n💾 Matrix scanning results successfully exported to 'content/cta_scan_results.csv'")

if __name__ == "__main__":
    run_screener()


✅ [SUCCESS]: Found source dataset at path: 'sp500.csv'
🔍 Initializing matrix scan across 515 tracking targets...


/tmp/ipykernel_1013/1686883287.py:117: FutureWarning: YF.download() has changed argument auto_adjust default to True
  d_data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipykernel_1013/1686883287.py:118: FutureWarning: YF.download() has changed argument auto_adjust default to True
  w_data = yf.download(ticker, start=start_date, end=end_date, interval="1wk", progress=False)
/tmp/ipykernel_1013/1686883287.py:119: FutureWarning: YF.download() has changed argument auto_adjust default to True
  m_data = yf.download(ticker, start=start_date, end=end_date, interval="1mo", progress=False)
/tmp/ipykernel_1013/1686883287.py:117: FutureWarning: YF.download() has changed argument auto_adjust default to True
  d_data = yf.download(ticker, start=start_date, end=end_date, progress=False)
/tmp/ipykernel_1013/1686883287.py:118: FutureWarning: YF.download() has changed argument auto_adjust default to True
  w_data = yf.download(ticker, start=start_date, end=end_date, inte


🔥 LIVE INSTITUTIONAL CTA TRADING DASHBOARD
No tickers currently registering an active Yellow or Orange dot setup today.
